In [131]:
# Importing the libraries that will be needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3 as sql

In [132]:
conn = sql.connect("download.db")

In [133]:
# Reading members table
members = pd.read_sql('SELECT * FROM MEMBERS', conn)
members.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23
3,1004,Fares,Wahba,7.0,Maadi,inactive,2024-10-09
4,1005,Youssef,Halim,9.0,Maadi,Active,2024-05-05


In [134]:
# Reading books table
books = pd.read_sql('SELECT * FROM BOOKS', conn)
books.head()

,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez


In [135]:
# Reading checkouts table
checkouts = pd.read_sql('SELECT * FROM CHECKOUTS', conn)
checkouts.head()

,checkout_id,member_id,book_id,checkout_date,return_date
0,9263,1047,517,2024-10-21,2024-11-07
1,9340,1072,513,2025-08-24,2025-09-01
2,9231,1053,523,2024-02-04,2024-02-16
3,9129,1032,513,2025-06-21,2025-06-29
4,9370,1079,511,2025-11-11,2025-12-03


In [136]:
# Making a function ro read sql
def sql_query(q):
    return pd.read_sql(q, conn)

In [137]:
# Counting the total checkouts for each member, use left join to show members with zero (0) checkouts
count = """
SELECT M.member_id, M.first_name, last_name, COUNT(C.checkout_id) AS Borrowing_COUNT
FROM MEMBERS M
LEFT JOIN CHECKOUTS C
ON M.member_id = C.member_id
GROUP BY M.member_id
"""
sql_query(count)

,member_id,first_name,last_name,Borrowing_COUNT
0,1001,Salma,Ibrahim,1
1,1002,Fares,Saleh,2
2,1003,Bassel,Hegazy,9
3,1004,Fares,Wahba,0
4,1005,Youssef,Halim,3
...,...,...,...,...
75,1076,Dina,Wahba,7
76,1077,Lina,Rashad,6
77,1078,Habiba,Osman,0
78,1079,Rana,Osman,10


In [138]:
# Showing the titles of books whose aauthor name starts with M 
like = """
SELECT title
FROM BOOKS
WHERE author like "M%"
"""
print("The titles of the books whose author name starts with M")
sql_query(like)

The titles of the books whose author name starts with M


,title
0,Storms and Sailboats
1,The Clockwork Orchard
2,Notes from the Delta
3,The Glass Beehive


In [139]:
# Showing the most 5 borrowed books with counting the checkouts
popular_books = """
SELECT B.book_id, B.title, COUNT(c.checkout_id) AS COUNT
FROM BOOKS B
JOIN CHECKOUTS C
ON B.book_id = C.book_id
GROUP BY B.book_id 
ORDER BY COUNT DESC LIMIT 5
"""
sql_query(popular_books)

,book_id,title,COUNT
0,501,The Silver Kite,57
1,507,Fossils and Fireflies,55
2,513,Circuits for Beginners,46
3,519,Kites Over Cairo,38
4,525,Storms and Sailboats,25


In [140]:
# Showing the most 10 active members with counting their checkouts
active_readers = """
SELECT M.member_id, M.first_name, last_name, COUNT(C.checkout_id) AS Borrowing_Count
FROM MEMBERS M
JOIN CHECKOUTS C
ON M.member_id = C.member_id
GROUP BY M.member_id
ORDER BY Borrowing_Count DESC LIMIT 10
"""
sql_query(active_readers)

,member_id,first_name,last_name,Borrowing_Count
0,1034,Aya,Wahba,25
1,1044,Sherif,Saleh,21
2,1008,Ziad,Saleh,19
3,1027,Mostafa,Fouad,18
4,1010,Nour,Nabil,18
5,1065,Adam,Fahmy,17
6,1024,Youssef,Hegazy,17
7,1018,Ahmed,Shafik,17
8,1047,Sara,Rashad,16
9,1030,Reem,Osman,16


In [141]:
# Showing Maadi activity'look from newest to oldest through past the ten most recent
maadi = """
SELECT M.member_id, M.first_name, M.neighborhood, C.checkout_date
FROM MEMBERS M
JOIN CHECKOUTS C
ON M.member_id = C.member_id
WHERE M.neighborhood = "Maadi"
ORDER BY C.checkout_date DESC
LIMIT 10 OFFSET 10
"""
print("Neighborhood is Maadi")
sql_query(maadi)

Neighborhood is Maadi


,member_id,first_name,neighborhood,checkout_date
0,1003,Bassel,Maadi,2025-09-04
1,1017,Adam,Maadi,2025-08-25
2,1008,Ziad,Maadi,2025-08-23
3,1003,Bassel,Maadi,2025-08-21
4,1018,Ahmed,Maadi,2025-08-19
5,1015,Hamza,Maadi,2025-08-08
6,1018,Ahmed,Maadi,2025-08-04
7,1013,Ziad,Maadi,2025-07-27
8,1003,Bassel,Maadi,2025-07-22
9,1009,Hassan,Maadi,2025-07-22


In [142]:
# Combine members table with checkout table, count the number of borrowed books for each member by using transform function
combined_sql = pd.merge(members, checkouts, on = "member_id", how = "left")
combined_sql["borrowed_books"] = combined_sql.groupby("member_id")["checkout_id"].transform("count")
combined_sql.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date,checkout_id,book_id,checkout_date,return_date,borrowed_books
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05,9025.0,525.0,2024-10-24,2024-11-07,1
1,1002,Fares,Saleh,9.0,Maadi,Active,None,9013.0,501.0,2024-02-16,2024-02-29,2
2,1002,Fares,Saleh,9.0,Maadi,Active,None,9095.0,507.0,2024-06-25,2024-07-07,2
3,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9051.0,513.0,2025-07-22,2025-08-18,9
4,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9068.0,501.0,2025-11-14,2025-12-03,9


In [143]:
# Showing the number of rows and columns in the combined_sql DataFrame
combined_sql.shape

(409, 12)

In [144]:
# Reading the json file
json = pd.read_json("download.json")
json.head()

,book_id,genre,pages,publication_year,publisher
0,501,Adventure,128,2017.0,Nile Press
1,502,Adventure,109,2018.0,Delta House
2,503,Historical,259,NaN,Nile Press
3,504,Science,319,2009.0,Cairo Young Readers
4,505,Historical,216,2024.0,Oasis Books


In [145]:
# Showing the number of rows and columns in the json
json.shape

(32, 5)

In [146]:
# Combining the combined_sql with the json by book_id
combined_json_sql = pd.merge(combined_sql, json, on = "book_id", how = "left")
combined_json_sql.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date,checkout_id,book_id,checkout_date,return_date,borrowed_books,genre,pages,publication_year,publisher
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05,9025.0,525.0,2024-10-24,2024-11-07,1,Adventure,297.0,2015.0,Oasis Books
1,1002,Fares,Saleh,9.0,Maadi,Active,None,9013.0,501.0,2024-02-16,2024-02-29,2,Adventure,128.0,2017.0,Nile Press
2,1002,Fares,Saleh,9.0,Maadi,Active,None,9095.0,507.0,2024-06-25,2024-07-07,2,Science,160.0,2024.0,Nile Press
3,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9051.0,513.0,2025-07-22,2025-08-18,9,Science,294.0,2021.0,Oasis Books
4,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9068.0,501.0,2025-11-14,2025-12-03,9,Adventure,128.0,2017.0,Nile Press


In [147]:
# Showing the number of rows and columns in the combined_json_sql
combined_json_sql.shape

(409, 16)

In [148]:
# Reading the html file and selecting the first table in the html file
html = pd.read_html("download.html")
html_table = html[0]
html_table.head()

,Member ID,Book ID,Checkout Date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


In [149]:
# Showing the number of rows and columns in the html_table
html_table.shape

(26, 3)

In [150]:
# Changing the column names in the html_table by making them lower case and replacing spaces with underscores
html_table.columns = html_table.columns.str.lower().str.replace(" ", "_")
html_table.head()

,member_id,book_id,checkout_date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


In [151]:
# Combining the combined_json_sql with the html_table
combined_data = pd.concat([combined_json_sql, html_table], ignore_index=True)
combined_data.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date,checkout_id,book_id,checkout_date,return_date,borrowed_books,genre,pages,publication_year,publisher
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05,9025.0,525.0,2024-10-24,2024-11-07,1.0,Adventure,297.0,2015.0,Oasis Books
1,1002,Fares,Saleh,9.0,Maadi,Active,None,9013.0,501.0,2024-02-16,2024-02-29,2.0,Adventure,128.0,2017.0,Nile Press
2,1002,Fares,Saleh,9.0,Maadi,Active,None,9095.0,507.0,2024-06-25,2024-07-07,2.0,Science,160.0,2024.0,Nile Press
3,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9051.0,513.0,2025-07-22,2025-08-18,9.0,Science,294.0,2021.0,Oasis Books
4,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9068.0,501.0,2025-11-14,2025-12-03,9.0,Adventure,128.0,2017.0,Nile Press


In [152]:
# Showing the number of rows and columns in the combined_data
combined_data.shape

(435, 16)

In [153]:
combined_data.columns

Index(['member_id', 'first_name', 'last_name', 'grade', 'neighborhood',
       'membership_status', 'join_date', 'checkout_id', 'book_id',
       'checkout_date', 'return_date', 'borrowed_books', 'genre', 'pages',
       'publication_year', 'publisher'],
      dtype='object')

In [154]:
# Saving the combined_data to a csv file
combined_data.to_csv("task1_combined_data.csv", index=False)

In [155]:
# Reading the combined_data from the csv file
df = pd.read_csv("task1_combined_data.csv")
df.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date,checkout_id,book_id,checkout_date,return_date,borrowed_books,genre,pages,publication_year,publisher
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05,9025.0,525.0,2024-10-24,2024-11-07,1.0,Adventure,297.0,2015.0,Oasis Books
1,1002,Fares,Saleh,9.0,Maadi,Active,NaN,9013.0,501.0,2024-02-16,2024-02-29,2.0,Adventure,128.0,2017.0,Nile Press
2,1002,Fares,Saleh,9.0,Maadi,Active,NaN,9095.0,507.0,2024-06-25,2024-07-07,2.0,Science,160.0,2024.0,Nile Press
3,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9051.0,513.0,2025-07-22,2025-08-18,9.0,Science,294.0,2021.0,Oasis Books
4,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9068.0,501.0,2025-11-14,2025-12-03,9.0,Adventure,128.0,2017.0,Nile Press


In [156]:
# Showing the name of the columns in the df
df.columns

Index(['member_id', 'first_name', 'last_name', 'grade', 'neighborhood',
       'membership_status', 'join_date', 'checkout_id', 'book_id',
       'checkout_date', 'return_date', 'borrowed_books', 'genre', 'pages',
       'publication_year', 'publisher'],
      dtype='object')

In [157]:
# Showing the number of rows and columns in the df
df.shape

(435, 16)

In [158]:
# Showing the data types and non-null values in the df
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 435 entries, 0 to 434
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   member_id          435 non-null    int64  
 1   first_name         409 non-null    object 
 2   last_name          409 non-null    object 
 3   grade              371 non-null    float64
 4   neighborhood       409 non-null    object 
 5   membership_status  409 non-null    object 
 6   join_date          403 non-null    object 
 7   checkout_id        391 non-null    float64
 8   book_id            417 non-null    float64
 9   checkout_date      417 non-null    object 
 10  return_date        326 non-null    object 
 11  borrowed_books     409 non-null    float64
 12  genre              391 non-null    object 
 13  pages              391 non-null    float64
 14  publication_year   357 non-null    float64
 15  publisher          391 non-null    object 
dtypes: float64(6), int64(1), o

In [159]:
# Showing the summary statistics of the df
df.describe()

,member_id,grade,checkout_id,book_id,borrowed_books,pages,publication_year
count,435.000000,371.000000,391.000000,417.000000,409.000000,391.000000,357.000000
mean,1041.117241,7.560647,9192.611253,513.443645,11.723716,201.969309,2017.560224
std,26.126819,1.152419,110.279771,8.976053,7.251737,73.974175,4.523334
min,1001.000000,6.000000,9001.000000,501.000000,0.000000,104.000000,2009.000000
25%,1020.000000,7.000000,9097.500000,507.000000,6.000000,134.000000,2014.000000
50%,1038.000000,8.000000,9193.000000,513.000000,11.000000,160.000000,2017.000000
75%,1061.000000,9.000000,9287.500000,519.000000,17.000000,294.000000,2021.000000
max,1201.000000,9.000000,9383.000000,532.000000,25.000000,338.000000,2024.000000


In [160]:
# Showing the data types of the columns in the df
df.dtypes

member_id              int64
first_name            object
last_name             object
grade                float64
neighborhood          object
membership_status     object
join_date             object
checkout_id          float64
book_id              float64
checkout_date         object
return_date           object
borrowed_books       float64
genre                 object
pages                float64
publication_year     float64
publisher             object
dtype: object

In [161]:
# Coppying the df to a new DataFrame called df_cleaned
df_cleaned = df.copy()

In [162]:
# Converting the checkout_date, join_date, and return_date columns to datetime format
df_cleaned["checkout_date"] = pd.to_datetime(df_cleaned["checkout_date"], format="mixed")
df_cleaned["join_date"] = pd.to_datetime(df_cleaned["join_date"], format="mixed")
df_cleaned["return_date"] = pd.to_datetime(df_cleaned["return_date"], format="mixed")

In [163]:
# Showing the data types of the columns in the df_cleaned after converting the date columns to datetime format
df_cleaned.dtypes

member_id                     int64
first_name                   object
last_name                    object
grade                       float64
neighborhood                 object
membership_status            object
join_date            datetime64[ns]
checkout_id                 float64
book_id                     float64
checkout_date        datetime64[ns]
return_date          datetime64[ns]
borrowed_books              float64
genre                        object
pages                       float64
publication_year            float64
publisher                    object
dtype: object

In [164]:
# Checking for missing values in the df_cleaned
df_cleaned.isna().sum()

member_id              0
first_name            26
last_name             26
grade                 64
neighborhood          26
membership_status     26
join_date             32
checkout_id           44
book_id               18
checkout_date         18
return_date          109
borrowed_books        26
genre                 44
pages                 44
publication_year      78
publisher             44
dtype: int64

In [165]:
# Filling the missing values in the first_name column with "unknown"
df_cleaned["first_name"] = df_cleaned["first_name"].fillna("Unknown")

In [166]:
# Filling the missing values in the last_name column with "unknown"
df_cleaned["last_name"] = df_cleaned["last_name"].fillna("Unknown")

In [167]:
# Filling the missing values in the grade column with the mode of the grade column
df_cleaned["grade"] = df_cleaned["grade"].fillna(df_cleaned["grade"].mode()[0])

In [168]:
# Filling the missing values in the neighborhood column with the mode of the neighborhood column
df_cleaned["neighborhood"] = df_cleaned["neighborhood"].fillna(df_cleaned["neighborhood"].mode()[0])

In [169]:
# Filling the missing values in the membership_status column with the mode of the membership_status column
df_cleaned["membership_status"] = df_cleaned["membership_status"].fillna(df_cleaned["membership_status"].mode()[0])

In [170]:
# Filling the missing values in the join_date column with the mode of the join_date column
df_cleaned["join_date"] = df_cleaned["join_date"].fillna(df_cleaned["join_date"].mode()[0])

In [171]:
# Filling the missing values in the checkout_id column with "unknown"
df_cleaned["checkout_id"] = df_cleaned["checkout_id"].fillna("unknown")

In [172]:
# Filling the missing values in the book_id column with "unknown"
df_cleaned["book_id"] = df_cleaned["book_id"].fillna("unknown")

In [173]:
# Filling the missing values in the checkout_date column with the mode of the checkout_date column
df_cleaned["checkout_date"] = df_cleaned["checkout_date"].fillna(df_cleaned["checkout_date"].mode()[0])

In [174]:
# Filling the missing values in the return_date column with "Not Returned"
df_cleaned["return_date"] = df_cleaned["return_date"].fillna("Not Returned")

In [175]:
# Filling the missing values in the borrowed_books column with 0
df_cleaned["borrowed_books"] = df_cleaned["borrowed_books"].fillna(0)

In [176]:
# Filling the missing values in the genre column with the mode of the genre column
df_cleaned["genre"] = df_cleaned["genre"].fillna(df_cleaned["genre"].mode()[0])

In [177]:
# Filling the missing values in the pages column with the median of the pages column
df_cleaned["pages"] = df_cleaned["pages"].fillna(df_cleaned["pages"].median())

In [178]:
# Filling the missing values in the publication_year column with the mode of the publication_year column
df_cleaned["publication_year"] = df_cleaned["publication_year"].fillna(df_cleaned["publication_year"].mode()[0])

In [179]:
# Filling the missing values in the publisher column with "unknown"
df_cleaned["publisher"] = df_cleaned["publisher"].fillna("unknown")

In [180]:
# Checking for missing values in the df_cleaned after filling the missing values
df_cleaned.isna().sum()

member_id            0
first_name           0
last_name            0
grade                0
neighborhood         0
membership_status    0
join_date            0
checkout_id          0
book_id              0
checkout_date        0
return_date          0
borrowed_books       0
genre                0
pages                0
publication_year     0
publisher            0
dtype: int64

In [181]:
# Checking for duplicate rows in the df_cleaned
df_cleaned.duplicated().sum()

np.int64(8)

In [182]:
# Dropping the duplicate rows in the df_cleaned
df_cleaned = df_cleaned.drop_duplicates()

In [183]:
# Checking for duplicate rows in the df_cleaned after dropping the duplicate rows
df_cleaned.duplicated().sum()

np.int64(0)

In [184]:
# Showing the number of rows and columns in the df_cleaned after dropping the duplicate rows
df_cleaned.shape

(427, 16)

In [185]:
# Showing the data types of the columns in the df_cleaned
df_cleaned.dtypes

member_id                     int64
first_name                   object
last_name                    object
grade                       float64
neighborhood                 object
membership_status            object
join_date            datetime64[ns]
checkout_id                  object
book_id                      object
checkout_date        datetime64[ns]
return_date                  object
borrowed_books              float64
genre                        object
pages                       float64
publication_year            float64
publisher                    object
dtype: object

In [186]:
# Showing the unique values in the first_name column to ensure data coinsistency 
df_cleaned["first_name"].unique()

array(['Salma', 'Fares', 'Bassel', 'Youssef', 'Layla', 'Sherif', 'Ziad',
       'Hassan', 'Nour', 'Menna', 'Ali', 'Habiba', 'Hamza', 'Dina',
       'Adam', 'Ahmed', 'Mostafa', 'Rana', 'Hana', 'Nada', 'Reem',
       'Marwan', 'Aya', 'Karim', 'Lina', 'Retaj', 'Mai', 'Amir', 'Sara',
       'Farida', 'Tarek', 'Jana', 'Seif', 'Yassin', 'Malak', 'Unknown'],
      dtype=object)

In [187]:
# Showing the unique values in the last_name column to ensure data coinsistency
df_cleaned["last_name"].unique()

array(['Ibrahim', 'Saleh', 'Hegazy', 'Wahba', 'Halim', 'Mansour', 'Fouad',
       'Nabil', 'Sabry', 'Rashad', 'Badr', 'Shafik', 'Adel', 'Kamel',
       'Osman', 'Fahmy', 'Zaki', 'Riad', 'Gamal', 'Unknown'], dtype=object)

In [188]:
# Showing the unique values in the neighborhood column to ensure data coinsistency
df_cleaned["neighborhood"].unique()

array(['Maadi', 'Maadi ', 'Nasr City', 'Nasr  City', 'NASR CITY',
       'HELIOPOLIS', 'Heliopolis', 'zamalek', 'Zamalek', 'Shubra'],
      dtype=object)

In [189]:
# Cleaning the neighborhood column by removing leading and trailing spaces,
# converting to title case, and replacing multiple spaces with a single space
clean_neighborhood = df_cleaned["neighborhood"].str.strip().str.title().str.split().str.join(" ")

In [190]:
# Counting the number of rows that were affected before the cleaning of the neighborhood column
affected_rows_n = (df_cleaned["neighborhood"] != clean_neighborhood).sum()
affected_rows_n

np.int64(31)

In [191]:
# Replacing the neighborhood column in the df_cleaned with the cleaned neighborhood column
df_cleaned["neighborhood"] = clean_neighborhood

In [192]:
# Showing the unique values in the neighborhood column after cleaning to ensure data coinsistency
df_cleaned["neighborhood"].unique()

array(['Maadi', 'Nasr City', 'Heliopolis', 'Zamalek', 'Shubra'],
      dtype=object)

In [193]:
# Showing the unique values in the membership_status column to ensure data coinsistency
df_cleaned["membership_status"].unique()

array(['Active', 'inactive', 'Inactive', 'active', 'INACTIVE'],
      dtype=object)

In [194]:
# Cleaning the membership_status column by removing leading and trailing spaces by using .str.strip()
# and converting to title case by using .str.title()
clean_membership_status = df_cleaned["membership_status"].str.strip().str.title()

In [195]:
# Counting the number of rows that were affected before the cleaning of the membership_status column
affected_rows_m = (df_cleaned["membership_status"] != clean_membership_status).sum()
affected_rows_m

np.int64(77)

In [196]:
# Replacing the membership_status column in the df_cleaned with the cleaned membership_status column
df_cleaned["membership_status"] = clean_membership_status

In [197]:
# Showing the unique values in the membership_status column after cleaning to ensure data coinsistency
df_cleaned["membership_status"].unique()

array(['Active', 'Inactive'], dtype=object)

In [198]:
# Showing the unique values in the genre column to ensure data coinsistency
df_cleaned["genre"].unique()

array(['Adventure', 'Science', 'Historical', 'Friendship', 'Mystery',
       'Nature', 'Science Fiction', 'Poetry'], dtype=object)

In [199]:
# Showing the unique values in the publisher column to ensure data coinsistency
df_cleaned["publisher"].unique()

array(['Oasis Books', 'Nile Press', 'Cairo Young Readers', 'unknown',
       'Delta House'], dtype=object)

In [200]:
# Checking for invalid member_id values in the df_cleaned by using the ~ operator to filter the rows in the df_cleaned where the member_id is not in the members table
invalid_members = df_cleaned[~df_cleaned["member_id"].isin(members["member_id"])]
invalid_members

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date,checkout_id,book_id,checkout_date,return_date,borrowed_books,genre,pages,publication_year,publisher
413,1104,Unknown,Unknown,9.0,Nasr City,Active,2025-01-22,unknown,515.0,2025-07-07,Not Returned,0.0,Science,160.0,2017.0,unknown
418,1201,Unknown,Unknown,9.0,Nasr City,Active,2025-01-22,unknown,509.0,2025-07-10,Not Returned,0.0,Science,160.0,2017.0,unknown
420,1104,Unknown,Unknown,9.0,Nasr City,Active,2025-01-22,unknown,526.0,2025-07-05,Not Returned,0.0,Science,160.0,2017.0,unknown
423,1150,Unknown,Unknown,9.0,Nasr City,Active,2025-01-22,unknown,530.0,2025-07-10,Not Returned,0.0,Science,160.0,2017.0,unknown
433,1201,Unknown,Unknown,9.0,Nasr City,Active,2025-01-22,unknown,523.0,2025-07-08,Not Returned,0.0,Science,160.0,2017.0,unknown


In [201]:
# Showing the number of invalid member_id values in the df
len(invalid_members)

5

In [202]:
# Dropping the rows in the df_cleaned where the member_id is not in the members table 
#by using the .isin() method to filter the rows in the df_cleaned where the member_id is in the members table
df_cleaned = df_cleaned[df_cleaned["member_id"].isin(members["member_id"])]

In [203]:
# Showing the first 5 rows of the df_cleaned to ensure that the data is clean
df_cleaned.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date,checkout_id,book_id,checkout_date,return_date,borrowed_books,genre,pages,publication_year,publisher
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05,9025.0,525.0,2024-10-24,2024-11-07 00:00:00,1.0,Adventure,297.0,2015.0,Oasis Books
1,1002,Fares,Saleh,9.0,Maadi,Active,2025-01-22,9013.0,501.0,2024-02-16,2024-02-29 00:00:00,2.0,Adventure,128.0,2017.0,Nile Press
2,1002,Fares,Saleh,9.0,Maadi,Active,2025-01-22,9095.0,507.0,2024-06-25,2024-07-07 00:00:00,2.0,Science,160.0,2024.0,Nile Press
3,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9051.0,513.0,2025-07-22,2025-08-18 00:00:00,9.0,Science,294.0,2021.0,Oasis Books
4,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9068.0,501.0,2025-11-14,2025-12-03 00:00:00,9.0,Adventure,128.0,2017.0,Nile Press


In [204]:
# Showing the number of rows and columns in the df_cleaned after dropping the invalid member_id values
df_cleaned.shape

(422, 16)

In [205]:
# Saving the cleaned data to a csv file
df_cleaned.to_csv("task2_cleaned_data.csv", index=False)